In [ ]:
# 1. Create traversal methods for the CategoryNode structure from Exercise 1:

# Part A: In-order Traversal (Left → Root → Right)


def in_order_collect(node):
    """
    Returns a list of category names in in-order sequence.
    Logic: Traverse Left subtree -> Visit Root -> Traverse Right subtree
    """
    if node is None:
        return []

    # Recursively collect from left, then current, then right
    return in_order_collect(node.left) + [node.name] + in_order_collect(node.right)


def in_order_accumulate_posts(node, current_total=0):
    """
    Processes categories in in-order while building a running total of posts.
    Returns the final accumulated total after traversing the tree.
    """
    if node is None:
        return current_total

    # 1. Traverse the left subtree and get the updated total
    current_total = in_order_accumulate_posts(node.left, current_total)

    # 2. Process current node: Add its post_count to the running total
    current_total += node.post_count
    print(f"Category: {node.name:15} | Running Total: {current_total}")

    # 3. Traverse the right subtree with the new total
    return in_order_accumulate_posts(node.right, current_total)


def in_order_find_kth(k, node):
    """
    Finds the k-th category in in-order sequence (1-indexed).
    """
    # Simple implementation: Collect all names first, then pick the k-th element
    sequence = in_order_collect(node)

    if 1 <= k <= len(sequence):
        return sequence[k - 1] # Subtract 1 because Python lists are 0-indexed

    return None

In [ ]:

# Part B: Pre-order Traversal (Root → Left → Right)

def pre_order_export(node, depth=0):
    """
    Generates a tree representation for storage/transmission.
    Formatted as a string with indentation based on tree depth.
    """
    if node is None:
        return ""

    # 1. Process Root: Add indentation (2 spaces per depth level)
    indent = "  " * depth
    line = f"{indent}{node.name}({node.post_count})\n"

    # 2. Recurse Left, then Right (incrementing depth)
    left_output = pre_order_export(node.left, depth + 1)
    right_output = pre_order_export(node.right, depth + 1)

    return line + left_output + right_output


def pre_order_copy(node):
    """
    Creates a deep copy of the entire tree.
    Returns a new CategoryNode object that is identical to the original.
    """
    if node is None:
        return None

    # 1. Process Root: Create a new instance of the current node
    new_node = CategoryNode(node.category_id, node.name, node.post_count)

    # 2. Recurse Left and Right: Assign new subtrees to the new node
    new_node.left = pre_order_copy(node.left)
    new_node.right = pre_order_copy(node.right)

    # Set parent reference if your Exercise 1 structure requires it
    if new_node.left: new_node.left.parent = new_node
    if new_node.right: new_node.right.parent = new_node

    return new_node


def pre_order_serialize(node):
    """
    Converts the tree to a flat string format.
    Example output: "Technology(150) | Programming(85) | Python(42) | ..."
    """
    if node is None:
        return ""

    # 1. Process Root: Get current node's info
    parts = [f"{node.name}({node.post_count})"]

    # 2. Recurse: Collect serialized strings from children
    left_str = pre_order_serialize(node.left)
    right_str = pre_order_serialize(node.right)

    # Add to parts list if children exist (to avoid empty pipes)
    if left_str: parts.append(left_str)
    if right_str: parts.append(right_str)

    return " | ".join(parts)

In [ ]:

# Part C: Post-order Traversal (Left → Right → Root)

def post_order_total_posts(node):
    """
    Computes total posts in a category including all subcategories.
    Formula: Left Subtree Total + Right Subtree Total + Current Node's Count
    """
    if node is None:
        return 0

    # 1. Get totals from children first (Post-order logic)
    left_total = post_order_total_posts(node.left)
    right_total = post_order_total_posts(node.right)

    # 2. Add children's totals to the current node's post_count
    return left_total + right_total + node.post_count


def post_order_collect_leaves(node):
    """
    Collects all leaf categories (nodes with no children).
    """
    if node is None:
        return []

    # If it's a leaf node (no left and no right), return it in a list
    if node.left is None and node.right is None:
        return [node.name]

    # Otherwise, combine leaf lists from both subtrees
    return post_order_collect_leaves(node.left) + post_order_collect_leaves(node.right)


def post_order_average_depth(node, current_depth=0):
    """
    Calculates the average depth of all leaf categories in the tree.
    Returns a float representing the average depth.
    """
    def get_leaf_depths(n, depth):
        if n is None:
            return []
        if n.left is None and n.right is None:
            return [depth]
        return get_leaf_depths(n.left, depth + 1) + get_leaf_depths(n.right, depth + 1)

    depths = get_leaf_depths(node, current_depth)

    if not depths:
        return 0.0

    return sum(depths) / len(depths)

In [ ]:
# 2. Implement traversal-based analytics:

def find_most_popular_category(node):
    """
    Finds the category with the highest post count (considering only the node itself).
    Uses a recursive approach to compare values across the entire tree.
    """
    if node is None:
        return None

    # 1. Assume the current node is the most popular so far
    best_node = node

    # 2. Recursively find the most popular nodes in the subtrees
    left_best = find_most_popular_category(node.left)
    right_best = find_most_popular_category(node.right)

    # 3. Compare current best with left subtree's best
    if left_best and left_best.post_count > best_node.post_count:
        best_node = left_best

    # 4. Compare current best with right subtree's best
    if right_best and right_best.post_count > best_node.post_count:
        best_node = right_best

    return best_node

In [ ]:
def category_with_most_subcategories(node):
    """
    Finds the category with the most direct children (0, 1, or 2).
    """
    if node is None:
        return None

    # Helper function to count direct children of a given node
    def get_child_count(n):
        count = 0
        if n.left: count += 1
        if n.right: count += 1
        return count

    # 1. Start with the current node's count
    best_node = node
    max_children = get_child_count(node)

    # 2. Check subtrees recursively to find nodes with potentially more children
    left_best = category_with_most_subcategories(node.left)
    right_best = category_with_most_subcategories(node.right)

    # 3. Compare current max with results from left subtree
    if left_best:
        left_count = get_child_count(left_best)
        if left_count > max_children:
            best_node = left_best
            max_children = left_count

    # 4. Compare current max with results from right subtree
    if right_best:
        right_count = get_child_count(right_best)
        if right_count > max_children:
            best_node = right_best
            max_children = right_count

    return best_node